<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/d4-finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 5 · Notebook 7 — Fine-tuning a pretrained model, and Grad-CAM

In [ ]:
# Installs what Colab doesn't ship with.
!pip install -q medmnist

Yesterday your team trained a CNN from nothing: every number started random.
A **pretrained** model has already learned from 1.2 million photos
(ImageNet), so its early layers already detect edges, textures and shapes.
**Fine-tuning** keeps those layers and retrains only the end for your task.
It needs far less data and time, which is exactly the situation of a capstone
project with a few thousand labelled slides.

Then we turn to histology, Project I's subject, and ask a question that
matters as much as accuracy: *where was the model looking?*

Same rules: 🔮 predict, ✏️ replace the `...`, ✅ check. Swap driver and
navigator at every `##`.

In [ ]:
import os
import hashlib
import pathlib
import shutil
import urllib.request
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def fetch_from_mirror(url, dest, sha256):
    """Download url to dest and check its sha256. Returns True if the file is good."""
    # The original data servers get slow when a whole class downloads at once,
    # so we keep a copy of each file on our own fast server.
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def good(path):
        with open(path, "rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == sha256

    if dest.exists() and good(dest):
        return True
    # Download to a .part file first, so a stopped download never looks finished.
    part = dest.with_name(dest.name + ".part")
    try:
        # Our server turns away requests that look like they come from a robot,
        # so we give ours a name.
        request = urllib.request.Request(url, headers={"User-Agent": "tk-accelerator-notebooks"})
        with urllib.request.urlopen(request, timeout=30) as reply, open(part, "wb") as out:
            shutil.copyfileobj(reply, out)
        if not good(part):
            raise ValueError("the checksum does not match")
        part.replace(dest)
        return True
    except Exception:
        part.unlink(missing_ok=True)
        print("Our copy of the data did not download, so we are using the original source.")
        return False


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_data(name, *paths):
    """Stop with a plain message unless the downloaded files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} data isn't downloaded yet. Run the ⬇️ Download cell above first.")


def require_model(name, *paths):
    """Stop with a plain message unless the downloaded model files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} isn't downloaded yet. Run the ⬇️ Download cell above first.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
weights = torchvision.models.ResNet18_Weights.DEFAULT
RESNET_FILE = pathlib.Path(torch.hub.get_dir()) / "checkpoints" / os.path.basename(weights.url)
torch.manual_seed(0)
print("Running on", device)

## 1. Borrow a trained brain

⬇️ **Download the model (45 MB).** Run the next cell when you start section 1. It downloads the ResNet-18 weights, 45 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too.

In [ ]:
# ⬇️ Download cell: ResNet-18 weights (45 MB)
if RESNET_FILE.exists():
    print("The ResNet-18 weights are already downloaded. Nothing to do.")
else:
    weights.get_state_dict(progress=True)
    print("Done: ResNet-18 weights downloaded (45 MB).")

In [ ]:
require_model("ResNet-18 weights file", RESNET_FILE)
model = torchvision.models.resnet18(weights=weights)
print(model.fc)

ResNet-18's last layer, `model.fc`, turns 512 features into 1,000 scores, one
per ImageNet class. CIFAR-10 has 10 classes, so that layer must be replaced.

✏️ **Exercise 1a.** Write `make_finetune_model(n_classes)`: load ResNet-18 with
`weights`, **freeze** every existing number, then replace `model.fc` with a
new `nn.Linear` from 512 features to `n_classes`. To freeze, loop over
`model.parameters()` and set each one's `requires_grad` to `False`.

In [ ]:
def make_finetune_model(n_classes):
    model = torchvision.models.resnet18(weights=weights)
    ...  # 1. freeze every parameter
    ...  # 2. replace model.fc
    return model

🔮 **Predict.** After freezing and replacing the head for 10 classes, how many
numbers are left to train? (The new layer has a weight for every
feature–class pair, plus one bias per class.)

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
m = make_finetune_model(10)
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
check(tuple(m(torch.zeros(2, 3, 64, 64)).shape) == (2, 10), "The model should give 10 scores per image. Did you replace model.fc?")
check(trainable > 0, "Nothing is trainable. Freeze first, then replace fc, so the new layer stays trainable.")
check(trainable < 100_000, f"{trainable:,} numbers would train. Did you freeze the backbone?")
print(f"Trainable numbers: {trainable:,} of {sum(p.numel() for p in m.parameters()):,}")

Everyone continues from here (code hidden; double-click to see it after).

In [ ]:
#@title Run this to continue { display-mode: "form" }
def make_finetune_model(n_classes):
    require_model("ResNet-18 weights file", RESNET_FILE)
    model = torchvision.models.resnet18(weights=weights)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(512, n_classes)
    return model


def train_head(model, loader, epochs=1):
    model.to(device).train()
    for layer in model.modules():
        if isinstance(layer, nn.BatchNorm2d) and not layer.weight.requires_grad:
            layer.eval()  # a frozen layer keeps its statistics frozen too
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device).view(-1)
            optimizer.zero_grad()
            loss_fn(model(images), labels).backward()
            optimizer.step()


def accuracy_on(model, loader):
    model.eval()
    right = total = 0
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.to(device)).argmax(dim=1).cpu()
            right += (preds == labels.view(-1)).sum().item()
            total += len(preds)
    return right / total

🔮 **Predict.** Yesterday your team trained a CNN from scratch, on up to
45,000 photos, for many epochs. Here we train only the new last layer, on
5,000 photos, for one epoch. Higher or lower validation accuracy than your
team's model? Closer than you'd expect?

In [ ]:
my_prediction = ""  # @param {type:"string"}

⬇️ **Download the data (170 MB).** Run the next cell before the training cell below. It downloads 170 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too.

In [ ]:
# ⬇️ Download cell: CIFAR-10 (170 MB)
import tarfile

if pathlib.Path("data/cifar-10-batches-py").exists():
    print("CIFAR-10 is already downloaded. Nothing to do.")
else:
    if fetch_from_mirror("https://data.tk.sg/cifar-10/cifar-10-python.tar.gz", "data/cifar-10-python.tar.gz",
                         "6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce"):
        # Unpack into a spare folder first, so a stopped unpack never looks finished.
        shutil.rmtree("data/cifar-unpacking", ignore_errors=True)
        with tarfile.open("data/cifar-10-python.tar.gz") as tar:
            tar.extractall("data/cifar-unpacking", filter="data")
        pathlib.Path("data/cifar-unpacking/cifar-10-batches-py").rename("data/cifar-10-batches-py")
        shutil.rmtree("data/cifar-unpacking")
    else:
        torchvision.datasets.CIFAR10("data", download=True)
    print("Done: CIFAR-10 downloaded (170 MB).")

In [ ]:
SIZE = 96  # ResNet was trained on large photos; CIFAR's 32×32 are enlarged
tf = transforms.Compose([transforms.Resize(SIZE), transforms.ToTensor(),
                         transforms.Normalize(weights.transforms().mean, weights.transforms().std)])
require_data("CIFAR-10", "data/cifar-10-batches-py")
cifar = torchvision.datasets.CIFAR10("data", train=True, download=False, transform=tf)
order = torch.randperm(50000, generator=torch.Generator().manual_seed(0))
n = 300 if QUICK else 5000
cifar_train = torch.utils.data.DataLoader(torch.utils.data.Subset(cifar, order[:n].tolist()), batch_size=128, shuffle=True)
cifar_val = torch.utils.data.DataLoader(torch.utils.data.Subset(cifar, order[45000:45000 + (300 if QUICK else 2000)].tolist()), batch_size=256)

started = time.time()
cifar_model = make_finetune_model(10)
train_head(cifar_model, cifar_train)
print(f"validation accuracy {accuracy_on(cifar_model, cifar_val):.1%} after {time.time() - started:.0f} s on {device}")

## 2. Histology: nine kinds of tissue

PathMNIST is a public set of 107,180 small patches cut from slides of
colorectal cancer, each labelled with one of nine tissue types. It is the
same kind of image as Project I, and one class is the tumour itself. (The
Project I data is different and stays inside the programme.)

⬇️ **Download the data (206 MB).** Run the next cell when you start section 2. It downloads 206 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too.

In [ ]:
# ⬇️ Download cell: PathMNIST (206 MB)
import medmnist

if pathlib.Path("data/pathmnist.npz").exists():
    print("PathMNIST is already downloaded. Nothing to do.")
else:
    if not fetch_from_mirror("https://data.tk.sg/pathmnist/pathmnist.npz", "data/pathmnist.npz",
                             "81823f52dc622e69db2db4c72f8e8e617938dd6864d3c1f23d4e49724a28ea72"):
        medmnist.PathMNIST(split="val", download=True, root="data")
    print("Done: PathMNIST downloaded (206 MB).")

In [ ]:
import medmnist

os.makedirs("data", exist_ok=True)
TISSUES = list(medmnist.INFO["pathmnist"]["label"].values())
SHORT = ["adipose", "background", "debris", "lymphocytes", "mucus",
         "muscle", "normal mucosa", "stroma", "TUMOUR"]
path_tf = transforms.Compose([transforms.Resize(112), transforms.ToTensor(),
                              transforms.Normalize(weights.transforms().mean, weights.transforms().std)])
require_data("PathMNIST", "data/pathmnist.npz")
path_train = medmnist.PathMNIST(split="train", download=False, root="data", transform=path_tf)
path_val = medmnist.PathMNIST(split="val", download=False, root="data", transform=path_tf)
raw = medmnist.PathMNIST(split="val", download=False, root="data")

fig, axes = plt.subplots(1, 9, figsize=(16, 2.4))
shown = set()
for img, label in raw:
    k = int(label[0])
    if k not in shown:
        axes[k].imshow(img)
        axes[k].set_title(TISSUES[k].replace(" ", "\n"), fontsize=8)
        axes[k].axis("off")
        shown.add(k)
    if len(shown) == 9:
        break
plt.show()

🔮 **Predict.** Which two tissue types do you think the model will confuse
most? And which one mistake would worry a doctor most?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
n = 500 if QUICK else 4000
sub = torch.randperm(len(path_train), generator=torch.Generator().manual_seed(0))[:n].tolist()
path_train_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(path_train, sub), batch_size=128, shuffle=True)
path_val_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(path_val, list(range(500 if QUICK else 2000))), batch_size=256)

started = time.time()
tissue_model = make_finetune_model(9)
train_head(tissue_model, path_train_loader)
print(f"validation accuracy {accuracy_on(tissue_model, path_val_loader):.1%} after {time.time() - started:.0f} s")

For a doctor, the mistakes are not equal. `TISSUES[8]` is the tumour
(colorectal adenocarcinoma epithelium); `TISSUES[6]` is normal colon mucosa.

✏️ **Exercise 2a.** Count `missed_tumours`: validation patches that are
really tumour (label 8) but the model called something else. Use
`predictions` and `truth` below.

In [ ]:
tissue_model.eval()
predictions, truth = [], []
with torch.no_grad():
    for images, labels in path_val_loader:
        predictions.append(tissue_model(images.to(device)).argmax(dim=1).cpu())
        truth.append(labels.view(-1))
predictions, truth = torch.cat(predictions), torch.cat(truth)
missed_tumours = ...

In [ ]:
check(is_done(missed_tumours), "Replace the ... .")
tumours = int((truth == 8).sum())
check(0 <= missed_tumours <= tumours, f"There are only {tumours} real tumour patches; your count must be between 0 and that.")
caught = int(((truth == 8) & (predictions == 8)).sum())
check(missed_tumours + caught == tumours, "Every real tumour patch is either caught or missed. Use truth == 8 and predictions != 8 together.")
print(f"{missed_tumours} of {tumours} tumour patches missed ({missed_tumours / tumours:.1%}).")

🤔 **Discuss.** A model that misses a tumour, and a model that raises a false
alarm on healthy tissue: which is worse, and what would you change in Project I
to make the worse one rarer?

## 3. Where was it looking? Grad-CAM

**Grad-CAM** colours the parts of an image that pushed the model towards its
answer. It looks at the last convolutional layers' feature maps and asks how
much each one changed the winning score. You don't need to write it; you need
to read it.

In [ ]:
def grad_cam(model, image, target=None):
    """Heatmap (0 to 1) of where `model` looked when scoring `target` for one image."""
    store = {}
    layer = model.layer3  # 7 × 7 regions for a 112 × 112 image
    h1 = layer.register_forward_hook(lambda m, i, o: store.update(act=o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: store.update(grad=go[0]))
    model.eval()
    x = image.unsqueeze(0).to(device).requires_grad_(True)
    scores = model(x)
    target = int(scores.argmax()) if target is None else target
    model.zero_grad()
    scores[0, target].backward()
    h1.remove()
    h2.remove()
    weights_per_map = store["grad"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights_per_map * store["act"]).sum(dim=1))[0]
    cam = F.interpolate(cam[None, None], size=image.shape[1:], mode="bilinear")[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), target


def show_cams(model, dataset, raw_dataset, indices):
    fig, axes = plt.subplots(2, len(indices), figsize=(2.4 * len(indices), 5))
    for col, i in enumerate(indices):
        image, label = dataset[i]
        cam, pred = grad_cam(model, image)
        picture = np.array(raw_dataset[i][0].resize((112, 112)))
        axes[0, col].imshow(picture)
        axes[0, col].set_title(f"#{i} true: {SHORT[int(label[0])]}", fontsize=8)
        axes[1, col].imshow(picture)
        axes[1, col].imshow(cam, cmap="jet", alpha=0.45)
        axes[1, col].set_title(f"said: {SHORT[pred]}", fontsize=8)
        for ax in axes[:, col]:
            ax.axis("off")
    plt.show()


show_class = 8  # @param {type:"integer"}
start = 0  # @param {type:"integer"}
indices = [i for i in range(len(path_val)) if int(path_val.labels[i][0]) == show_class][start:start + 6]
show_cams(tissue_model, path_val, raw, indices)

🔮 **Right for the wrong reason?** Look at the heatmaps for correct answers.
Is the model looking at the cells, or at the edge of the patch, or at empty
background? Page through more patches by changing `start` (and `show_class`:
8 is tumour, 6 normal mucosa) and re-running the cell. Pick the patch number
that worries you most and say why.

In [ ]:
worrying_patch = ""  # @param {type:"string"}
why = ""  # @param {type:"string"}

## Extension (if you finish early)

Open the Day 5 extension track in Sidecar, or:

- Unfreeze `model.layer4` as well (set its parameters' `requires_grad` back
  to `True`) and compare validation accuracy and training time.
- Train the PathMNIST model for 3 epochs. Does `missed_tumours` go down?
- Change `grad_cam` to use `model.layer4`. What happens to the heatmaps, and
  why?

In [ ]:
# Your extension work here.